In [2]:
import os
from pathlib import Path

import numpy as np
import scipy.io

from scipy.signal import butter, filtfilt, savgol_filter
from PIL import Image

In [3]:
PROJECT_DIR = Path("../Raw_Images")

DATA_DIR = Path("../../Data/normal")

GASF_DIR = PROJECT_DIR / "GASF"
GADF_DIR = PROJECT_DIR / "GADF"

print("Data directory:", DATA_DIR)
print("Exists:", DATA_DIR.exists())
print("Data directory:", PROJECT_DIR)
print("Exists:", PROJECT_DIR.exists())

Data directory: ..\..\Data\normal
Exists: True
Data directory: ..\Raw_Images
Exists: True


In [ ]:
def load_signal(file_path):

    data = scipy.io.loadmat(file_path)

    signal = None

    for key in data.keys():

        if key.endswith("_DE_time"):
            signal = data[key].flatten()
            break

    if signal is None:
        raise ValueError(f"No DE signal found in {file_path}")

    return signal.astype(np.float64)

In [7]:
def butterworth(signal, cutoff, fs, order=4):

    min_cut = fs / 2

    if cutoff >= min_cut:
        raise ValueError(
            f"Cutoff {cutoff} Hz must be lower than Nyquist frequency {min_cut} Hz"
        )

    b, a = butter(order, cutoff/min_cut,btype="high", analog=False)

    filtered = filtfilt(b, a, signal)

    return filtered

In [8]:
def s_enhancement(signal, k=6):

    signal = np.asarray(signal, dtype=np.float64)

    enhanced = np.zeros_like(signal)

    enhanced[1:-1] = ((signal[1:-1] - signal[:-2])**2 + (signal[1:-1] - signal[2:])**2)**(1/k)

    enhanced[0] = enhanced[1]
    enhanced[-1] = enhanced[-2]

    return enhanced

In [9]:
def savitzky_golay(signal):

    return savgol_filter(signal,window_length=11,polyorder=3)

In [10]:
def min_max_normalize(signal):

    min_val = np.min(signal)
    max_val = np.max(signal)

    if max_val == min_val:
        return np.zeros_like(signal)

    normalized = (2*(signal - min_val)/(max_val - min_val)) - 1

    return normalized

In [11]:
def polar_signal(signal):

    return np.arccos(signal)

In [12]:
def create_gasf(phi):

    gasf = np.cos(phi[:, None] + phi[None, :])

    return gasf

In [13]:
def create_gadf(phi):

    gadf = np.sin(phi[:, None] - phi[None, :])

    return gadf

In [ ]:
# WINDOW_SIZE = 1024

TRAIN_STEP = 512   
VAL_STEP = 1024
TEST_STEP = 1024       

# TRAIN_RATIO = 0.70
# VAL_RATIO = 0.15
# TEST_RATIO = 0.15

# FS = 12000
CUTOFF = 5000
FS = 48000
CUTOFF = 22000

WINDOW_SIZE = 1024
TRAIN_STEP = 512
VAL_STEP = 1024
TEST_STEP = 1024

In [53]:
def get_class_name(file_path, parent_folder):

    filename = Path(file_path).stem.upper()

    if parent_folder == "Ball fault":

        if filename.startswith("B007"):
            return "B_007"

        elif filename.startswith("B014"):
            return "B_014"

        elif filename.startswith("B021"):
            return "B_021"

    elif parent_folder == "Inner race":

        if filename.startswith("IR007"):
            return "IR_007"

        elif filename.startswith("IR014"):
            return "IR_014"

        elif filename.startswith("IR021"):
            return "IR_021"

    elif parent_folder == "Outer race":

        if filename.startswith("OR007"):
            return "OR_007"

        elif filename.startswith("OR014"):
            return "OR_014"

        elif filename.startswith("OR021"):
            return "OR_021"

    elif parent_folder.lower() == "normal":

        return "Normal"

    raise ValueError(
        f"Could not determine class for {file_path}"
    )

In [15]:
CLASSES = [
    "B_007",
    "B_014",
    "B_021",
    "IR_007",
    "IR_014",
    "IR_021",
    "OR_007",
    "OR_014",
    "OR_021",
    "Normal"
]

SPLITS = ["train", "val", "test"]

for base_dir in [GASF_DIR, GADF_DIR]:

    for split in SPLITS:

        for class_name in CLASSES:

            folder = base_dir / split / class_name

            folder.mkdir(
                parents=True,
                exist_ok=True
            )

print("Output folders created.")

Output folders created.


In [54]:
def get_windows(signal, is_normal=False):

    n = len(signal)

    train_end = int(0.70 * n)
    val_end = int(0.85 * n)

    splits = {}

    if is_normal:

        train_windows = []

        max_start = train_end - WINDOW_SIZE

        train_starts = np.linspace(0, max_start, 165,dtype=int
        )

        for start in train_starts:

            window = signal[start:start + WINDOW_SIZE]

            train_windows.append(window)

        splits["train"] = train_windows

        val_windows = []

        val_start = train_end
        max_start = val_end - WINDOW_SIZE

        val_starts = np.linspace(
            val_start,
            max_start,
            17,
            dtype=int
        )

        for start in val_starts:

            window = signal[start:start + WINDOW_SIZE]

            val_windows.append(window)

        splits["val"] = val_windows

        test_windows = []

        test_start = val_end
        max_start = n - WINDOW_SIZE

        test_starts = np.linspace(
            test_start,
            max_start,
            17,
            dtype=int
        )

        for start in test_starts:

            window = signal[start:start + WINDOW_SIZE]

            test_windows.append(window)

        splits["test"] = test_windows

    else:
        train_windows = []

        for start in range(0,train_end - WINDOW_SIZE + 1,TRAIN_STEP):

            window = signal[start:start + WINDOW_SIZE]

            train_windows.append(window)

        splits["train"] = train_windows

        val_windows = []

        for start in range(train_end,val_end - WINDOW_SIZE + 1,VAL_STEP):

            window = signal[start:start + WINDOW_SIZE]

            val_windows.append(window)

        splits["val"] = val_windows

        test_windows = []

        for start in range(val_end,n - WINDOW_SIZE + 1,TEST_STEP):

            window = signal[start:start + WINDOW_SIZE]

            test_windows.append(window)

        splits["test"] = test_windows

    return splits

In [55]:
def preprocess_window(window):    

    processed = min_max_normalize(window)

    pol = polar_signal(processed)

    return pol

In [56]:
def save_gaf_image(gaf, output_path):

    image = ((gaf + 1) * 127.5).clip(0, 255).astype(np.uint8)

    image = Image.fromarray(image,mode="L")

    image = image.resize((224, 224),Image.Resampling.LANCZOS)

    image.save(output_path)

In [57]:
def process_mat_file(mat_file):

    mat_file = Path(mat_file)

    parent_folder = mat_file.parent.name

    class_name = get_class_name(mat_file,parent_folder)

    print(f"\nProcessing: {mat_file.name}")

    print(f"Class: {class_name}")

    signal = load_signal(mat_file)

    print(f"Signal length: {len(signal):,}")

    is_normal = class_name == "Normal"

    splits = get_windows(signal,is_normal=is_normal)

    for split_name, windows in splits.items():

        print(f"{split_name}: {len(windows)} windows")

        for idx, window in enumerate(windows):

            smooth = preprocess_window(window)

            gasf = create_gasf(smooth)

            gasf_path = (GASF_DIR / split_name / class_name / f"{mat_file.stem}_{idx:04d}.png")

            save_gaf_image(gasf,gasf_path)
            
            gadf = create_gadf(smooth)

            gadf_path = (GADF_DIR / split_name / class_name / f"{mat_file.stem}_{idx:04d}.png")

            save_gaf_image(gadf,gadf_path)

In [ ]:
print("Image size:",img.size)
print("Mode:", img.mode)
print("Channels:", len(img.getbands()))
print("Size:", img.size)

In [59]:
def count_windows(signal_length):
    train_end = int(0.70 * signal_length)
    val_end = int(0.85 * signal_length)

    train_count = max(0,len(range(0,train_end - WINDOW_SIZE + 1,TRAIN_STEP)))

    val_count = max(0,len(range(train_end,val_end - WINDOW_SIZE + 1,VAL_STEP)))

    test_count = max(0,len(range(val_end,signal_length - WINDOW_SIZE + 1,TEST_STEP)))

    return train_count, val_count, test_count

In [60]:
results = []

for mat_file in all_mat_files:

    signal = load_signal(mat_file)

    train_count, val_count, test_count = count_windows(len(signal))

    class_name = get_class_name(mat_file,mat_file.parent.name)

    results.append({
        "file": mat_file.name,
        "class": class_name,
        "length": len(signal),
        "train": train_count,
        "val": val_count,
        "test": test_count
    })

In [ ]:
class_counts = (df_counts.groupby("class")[["train", "val", "test"]].sum())

print(class_counts)

In [63]:
class_counts["total"] = class_counts.sum(axis=1)

print(class_counts)

        train  val  test  total
class                          
B_007     660   68    68    796
B_014     660   68    68    796
B_021     660   68    68    796
IR_007    661   69    69    799
IR_014    660   68    68    796
IR_021    660   68    68    796
OR_007    661   68    68    797
OR_014    660   68    68    796
OR_021    662   68    68    798


In [64]:
# for i, mat_file in enumerate(all_mat_files):

#     print(f"\n{'='*60}")

#     print(f"File {i+1}/{len(all_mat_files)}")

#     print(f"{mat_file}")

#     process_mat_file(mat_file)

normal_files = [
    DATA_DIR / "97.mat",
    DATA_DIR / "98.mat",
    DATA_DIR / "99.mat",
    DATA_DIR / "100.mat"
]

for mat_file in normal_files:

    process_mat_file(mat_file)


Processing: 97.mat
Class: Normal
Signal length: 243,938
train: 165 windows
val: 17 windows
test: 17 windows

Processing: 98.mat
Class: Normal
Signal length: 483,903
train: 165 windows
val: 17 windows
test: 17 windows

Processing: 99.mat
Class: Normal
Signal length: 483,903
train: 165 windows
val: 17 windows
test: 17 windows

Processing: 100.mat
Class: Normal
Signal length: 485,643
train: 165 windows
val: 17 windows
test: 17 windows
